# More in Depth Tutorial Focused on Hydrological Analysis 

In [ ]:
#importing wbEnviornment
#   wbEnviornment contains all of the functions related to geospatial analysis, as well as the functions for reading and writing data. 
from whitebox_workflows import WbEnvironment, download_sample_data

wbe = WbEnvironment()
print(wbe.version()) #prints version number 

## Creating DEM from a LiDAR Point Cloud 

In [ ]:
wbe.verbose = True #allows various 'wbe' functions to output to terminal 
wbe.working_directory = download_sample_data('mill_brook') #downloading sample data provided from library
print(f'Data have been stored in: {wbe.working_directory}') 

In [ ]:
lidar = wbe.read_lidar('mill_brook.laz')
print(f"There are {lidar.header.number_of_points} points in the lidar dataset.")

In [ ]:
#incorporating lidar data to a raster DEM using triangulation 

#creating a DEM 
dem = wbe.lidar_tin_gridding(lidar, returns_included='all', cell_size=1.0, excluded_classes=[1], max_triangle_edge_length=100.0)

#fill missing data 
dem = wbe.fill_missing_data(dem, filter_size=35)

hs = wbe.multidirectional_hillshade(dem)
#wbe.write_raster(raster_obj, "output.tif")
wbe.write_raster(hs, 'hillshade.tif', compress=False) #compression is good, but it is a lot slower so here we wont use it


In [ ]:
#Smooth the DEM. This step normally takes some experimentation to get the parameters right, which is why 
# I save the raw DEM/hillshade. Comparison on the hillshade images allows me to tweak the parameters
# until I find that the output DEM has the appropriate level of smoothing that I need for my application.

dem_smoothed = wbe.feature_preserving_smoothing(dem, filter_size=11, normal_diff_threshold=45.0, iterations=5)
#Can save the smoothed DEM if one desires...

#Certainly the hillshade image will have to be saved in order to compare with the hillshade from the raw DEM to evaluate whether the smoothing was sufficient. 

hs = wbe.multidirectional_hillshade(dem_smoothed)
wbe.write_raster(hs, 'hillshade_smoothed.tif', compress=False)

In [ ]:
contours = wbe.contours_from_raster(dem_smoothed, contour_interval=10.0)
wbe.write_vector(contours, 'contours.shp')

## Performing Hydrological Analysis on the DEM 

In [ ]:
import math #Using log function below 

# Remove the depressions, first by breaching the depressions using a max dist s othat it doesnt carve excessively long trenches for very deep pits, and then filling the remaining depressions

dem_no_deps = wbe.breach_depressions_least_cost(dem_smoothed, flat_increment=0.001, max_dist=1000) #Max dist parameter changes according to DEM 
dem_no_deps = wbe.fill_depressions(dem_no_deps, flat_increment=0.001)

# performing a flow-accumulation operation. Here the Qin (2007) multiple flow direction algorithm is being used.
# there ae many other options available though, like the D-infinity

# stream channels are usually identified as areas of relatively high flow accumulation and are mapped by thresholding flow accumulation values. 
channel_threshold = 2500.0
flow_accum = wbe.qin_flow_accumulation(dem_no_deps, out_type='cells', convergence_threshold=channel_threshold, log_transform=True)
wbe.write_raster(flow_accum, 'qin_flow_accum.tif')

# map the streams by thresholding the flow accum raster, using the same convergence threshold used above. This way we can be assured that the streams are single-cell wide DB representation, which is needed for any stream network analysis operations
streams = flow_accum > math.log(channel_threshold)

In [ ]:
# extracting watershed for a specific outlet point 
outlet = wbe.read_vector('outlet.shp') # this is a vector point that was included when we downloaded the 'mill_break' dataset

# make sure the outlet is positioned along the stream 
outlet = wbe.jenson_snap_pour_points(outlet, streams, 5.0)

# we need a d8-pointer raster to be able to route flow through the network 
d8_pntr = wbe.d8_pointer(dem_no_deps)

# extract the outlet's watershed 
watershed = wbe.watershed(d8_pointer=d8_pntr, pour_points=outlet)

# vectorize the watershed 
watershed_vec = wbe.raster_to_vector_polygons(watershed)
# smooth the watershed map for visualization
watershed_vec = wbe.smooth_vectors(watershed_vec, filter_size = 5)
wbe.write_vector(watershed_vec, 'watershed.shp')

# now, we only want the stream s inside the watershed 
streams = streams * watershed 

# perform a stream network analysis on the stream vector 
streams_vec = wbe.raster_streams_to_vector(streams, d8_pntr)
streams_vec, tmp1, tmp2, tmp3 = wbe.vector_stream_network_analysis(streams_vec, dem_no_deps) # only want streams output 
wbe.write_vector(streams_vec, 'stream.shp')

# extract all of the watersheds, draining to each outlet on the edge of the DEM using the 'basins' function
basins = wbe.basins(d8_pntr)
wbe.write_raster(basins, 'basins.tif')

# extracting subcatchments, i.e. the area draining directly to each link in the stream network?
subcatchments = wbe.subbasins(d8_pntr, streams)
wbe.write_raster(subcatchments, 'subcatchments.tif')

strahler_basins = wbe.strahler_order_basins(d8_pointer=d8_pntr, streams=streams)
wbe.write_raster(strahler_basins, 'strahler_basins.tif')